# Opis

Krótki notebook, który pozwala przetestować działanie różnych elementów implementacyjnych w szybki sposób.

# Importy

In [20]:
IS_NEW_APPROACH = True
IS_VGAE = False

In [21]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import GradientAccumulationScheduler
import yaml
import sys
import tqdm
import wandb
import json
from pyprojroot import here

sys.path.append('../') # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.KlejdaGraphAutoencoder import KlejdaGraphAutoencoder
from src.models.KlejdaGAE.KlejdaVariationalGraphAutoencoder import KlejdaVariationalGraphAutoencoder
from src.models.NewGAE.GraphAutoencoder import GraphAutoencoder
from src.models.NewGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import os
import sys
current_dir = os.getcwd()
framspy_path = os.path.abspath(os.path.join(current_dir, '..', 'external', 'framspy'))
if framspy_path not in sys.path:
	sys.path.insert(0, framspy_path)
with open("../configs/final_evolution_config.yaml", 'r') as f:
	evolution_config = yaml.safe_load(f)
import frams

frams.init(
    evolution_config['frams_path']
)

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data



# Przetwarzanie

## Załadowanie danych i przygotowanie do przetwarzania

In [26]:
# dataset = FramsticksDummyDataset(num_samples=1000)
torch.set_float32_matmul_precision('high')
genotypes = []
with open("../results/sampled_best_individuals_30_numjoints_merged.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line.strip())
        genotypes.append(obj)

project_dir = here()
configs_dir = project_dir / 'configs'
if IS_NEW_APPROACH:
	config_gae_path = configs_dir / 'gae_config.yaml'
else:
	config_gae_path = configs_dir / 'klejda_gae_config.yaml'

with open(config_gae_path) as f:
    config = yaml.safe_load(f)

dataset = FramsticksGraphDataset(genotypes,config["max_nodes"])

dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Akumulator, mający za zadanie zmieniać rozmiar batcha wraz z postępującym uczeniem modelu
# W początkowej fazie rozmiar jest mniejszy, wtedy bowiem model lepiej naucza się jak torzyć macierze X i A
# w późniejszym etapie rozmiar batcha wzrasta, aby skupić się poprawnym zmniejszeniu locality loss

if IS_NEW_APPROACH:
	accumulator = GradientAccumulationScheduler(scheduling={
	    0:1,
		20:2,
		60:4,
		120:8,
		140:16,
	})
else:
	accumulator = GradientAccumulationScheduler(scheduling={
	    0:1,
	})

train_dataloader = DataLoader(
    train_dataset,
	# TODO: ile ustawić? Może też powinno być w configs, tak jak wszystko inne?
    batch_size=64,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
	drop_last=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    persistent_workers=True,
	drop_last=True
)

wandb.login()

True

## GAE

In [27]:
if IS_NEW_APPROACH:
	modelGAE = GraphAutoencoder(config, frams_module=frams)
else:
	modelGAE = KlejdaGraphAutoencoder(config, frams_module=frams)
wandb.finish()

In [28]:
if IS_NEW_APPROACH:
	run_name = "GAE_new_test"
else:
	run_name = "GAE_klejda_test"
wandb_logger = WandbLogger(project="Framsticks-GAE", name=run_name, save_dir = config["save_dir"])

trainer = pl.Trainer(
	precision="bf16-mixed",
	callbacks=[accumulator],
    max_epochs=160,
    logger=wandb_logger,
    log_every_n_steps=5,
    accelerator="auto",
    devices=1
)
trainer.fit(modelGAE, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
wandb.finish()

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
wandb: setting up run 28wy8axt
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260827_144209-28wy8axt
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run GAE_new_test
wandb:  View project at https://wandb.ai/witekadrian7-none/Framsticks-GAE
wandb:  View run at https://wandb.ai/witekadrian7-none/Framsticks-GAE/runs/28wy8axt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │ 71.6 K │ train │     0 │
│ 1 │ fc_z             │ Linear   │    975 │ train │     0 │
│ 2 │ decoder_a        │ DecoderA │ 39.0 K │ train │     0 │
│ 3 │ decoder_x        │ DecoderX │ 33.8 K │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 145 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 145 K                                                                                                
Total estimated model params size (MB): 0.581                                                                      
Modules in train mode: 73                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\pytorch_lightning\utilities\data.py:79: Trying to
infer the `batch_size` from an ambiguous collection. The batch size we found is 64. To avoid any miscalculations, 
use `self.log(..., batch_size=batch_size)`.

`Trainer.fit` stopped: `max_epochs=160` reached.


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:                      epoch ▁▁▁▁▁▂▂▂▂▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇█████
wandb: train/locality_correlation ▁▂▃▄▄▅▅▆▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇█▇█▇▇███████
wandb:               train/loss_A █▆▅▅▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:               train/loss_X █▇▆▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        train/loss_locality █▅▄▄▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:           train/loss_total █▅▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        trainer/global_step ▁▂▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██████████████
wandb:   val/locality_correlation ▁▃▆▆▇▇▇▇▇▇▇▇▇▇█▇██▇█████████████████████
wandb:                 val/loss_A █▅▄▄▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:                 val/loss_X █▆▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:                         +2 ...
wandb: 
wandb: Run summary:
wandb:                      epoch 159
wandb: train/locality_correlat

## VGAE

In [8]:
if IS_NEW_APPROACH:
	modelVGAE = VariationalGraphAutoencoder(config, frams_module=frams)
else:
	modelVGAE = KlejdaVariationalGraphAutoencoder(config, frams_module=frams)
wandb.finish()

In [7]:
if IS_NEW_APPROACH:
	run_name = "VGAE_new_test"
else:
	run_name = "VGAE_klejda_test"
wandb_logger = WandbLogger(project="Framsticks-VGAE", name=run_name, save_dir = config["save_dir"])
trainer = pl.Trainer(
    max_epochs=160,
    logger=wandb_logger,
    log_every_n_steps=5,
    accelerator="auto",
    devices=1
)

trainer.fit(modelVGAE, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
wandb.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
wandb: setting up run ubzfdz5t
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260827_081149-ubzfdz5t
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run VGAE_new_test
wandb:  View project at https://wandb.ai/witekadrian7-none/Framsticks-VGAE
wandb:  View run at https://wandb.ai/witekadrian7-none/Framsticks-VGAE/runs/ubzfdz5t
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │ 71.6 K │ train │     0 │
│ 1 │ fc_mu            │ Linear   │    975 │ train │     0 │
│ 2 │ fc_logvar        │ Linear   │    975 │ train │     0 │
│ 3 │ decoder_a        │ DecoderA │ 39.0 K │ train │     0 │
│ 4 │ decoder_x        │ DecoderX │ 37.7 K │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 150 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 150 K                                                                                                
Total estimated model params size (MB): 0.601                                                                      
Modules in train mode: 74                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=160` reached.


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading config.yaml
wandb: 
wandb: Run history:
wandb:                      epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
wandb: train/locality_correlation ▁▂▃▃▃▄▅▅▅▆▅▆▇▆▆▇▇▇▇▇▇▇▇▇▇▇█▇▇██▇██▇█████
wandb:               train/loss_A █▇▇▅▅▅▅▄▄▄▄▄▃▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁
wandb:              train/loss_KL ▂▁▂▇▇▄▄▄▁█▇▅▅▄▃▄▅▄▅▄▅▆▆▅▅▄▅▇▇▅▆▅▅▃▅▅▅▅▅▅
wandb:               train/loss_X ██▆▆▆▅▄▄▄▄▄▄▄▃▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁
wandb:        train/loss_locality █▇▆▅▅▅▄▄▅▅▄▄▃▄▃▃▃▃▂▂▂▂▂▂▁▂▂▁▁▁▁▂▁▁▁▁▁▁▁▂
wandb:           train/loss_total █▆▆▅▅▄▅▄▄▄▃▃▃▃▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▂▂▂▂▂▂▁▁▁▁▁
wandb:        trainer/global_step ▁▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇██
wandb:   val/locality_correlation ▁▃▅▅▅▄▇▆▅▇▇▇▆█▇▇▇█▇▇██▇▇▇█████▇█████▇███
wandb:                 val/loss_A █▇▅▅▅▅▅▄▅▄▃▄▃▃▃▃▃▃▂▃▂▂▂▂▃▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁
wandb:                         +4 ...
wandb: 
wandb: Run summary:
wandb:                      epoch 159
w